# 02 · Ask the catalog a scientific question

**Goal:** filter the same local records that drive the poster, preserve unknown values, and compare measurements with their contexts. The code uses the Python standard library plus the notebook's built-in display support.

Facts are curated summaries of official documentation, not observations downloaded from satellites. All records include citations and a verification date.

In [ ]:
from eo_guide import load_catalog, query, comparison
catalog = load_catalog()
for family in sorted({s['type'] for s in catalog['satellites']}):
    print(family, ':', ', '.join(s['name'] for s in query(catalog, family=family)))

In [ ]:
# Change property_name or agency to investigate another capability.
selected = query(catalog, property_name='surface reflectance')
[(s['name'], s['agency'], s['status']) for s in selected]

In [ ]:
comparison(catalog, ['landsat-8', 'sentinel-1c', 'enmap', 'goes-19'])

## Keep uncertainty visible

`None` means **not curated**. A geostationary imager's scan interval is deliberately not stored as an orbital repeat. Likewise, an observation plan or a cloud-free revisit cannot be deduced from orbital period.

In [ ]:
unknown = [s['name'] for s in catalog['satellites'] if s['orbit']['repeat_days'] is None]
print('No numeric repeat curated:', unknown)
mission = next(s for s in catalog['satellites'] if s['id'] == 'landsat-8')
for field, source_ids in mission['evidence'].items():
    print(field)
    for sid in source_ids:
        source = catalog['sources'][sid]
        print(' ', source['publisher'], source['url'], 'accessed', source['accessed'])

## Export a contextual comparison

This export is a local file in the notebook filesystem. In JupyterLite, right-click it in the file browser to download it. Notebook browser storage is separate from the checked-out repository.

In [ ]:
import csv
from pathlib import Path
export_path = Path('my-sensor-comparison.csv')
with export_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream)
    writer.writerow(['mission', 'measurement', 'spatial_context', 'temporal_context', 'source_urls'])
    for s in selected:
        for instrument in s['sensors']:
            writer.writerow([s['name'], '; '.join(instrument['observes']), instrument['spatial'], s['orbit']['temporal_note'], '; '.join(catalog['sources'][sid]['url'] for sid in instrument['sources'])])
print('Saved', export_path)

## Exercise

Compare a flooding question with a crop-stress question. Which measurement is useful, which product needs preprocessing, and what field observations are needed? Do not rank platforms only by their smallest pixel number.

Sources: [USGS Landsat 8](https://www.usgs.gov/landsat-missions/landsat-8), [Sentinel-1 mission](https://sentiwiki.copernicus.eu/web/s1-mission), [EnMAP mission](https://www.enmap.org/mission/) and [NOAA ABI products](https://www.ncei.noaa.gov/products/goes-terrestrial-weather-abi-glm); the catalog contains the complete record-level registry.